In [ ]:
import numpy as np
import pandas as pd
import os
import random
import copy

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

# 随机种子
SEED = 114514

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())
print("Device:", device)


PyTorch version: 2.12.1+cu132
CUDA version: 13.2
cuDNN version: 92000
Device: cuda


In [ ]:
sample_submission_path = "./data/sample_submission.csv"
test_features_path = "./data/test_features.csv"
train_drug_path = "./data/train_drug.csv"
train_features_path = "./data/train_features.csv"
train_targets_scored_path = "./data/train_targets_scored.csv"
train_targets_nonscored_path = "./data/train_targets_nonscored.csv"

# loading file function
def loading_file(path):
    if os.path.exists(path):
        print(f"get file {path}")
        try:
            data = pd.read_csv(path)
            print(data.shape)
            return data
        except Exception as e:
            print(f"error: {e}")
            return None
    else:
        print(f"couldn't find file: {path}")
        return None

sample_submission = loading_file(sample_submission_path)
test_features = loading_file(test_features_path)
train_drug = loading_file(train_drug_path)
train_features = loading_file(train_features_path)
train_targets_scored = loading_file(train_targets_scored_path)
train_targets_nonscored = loading_file(train_targets_nonscored_path)


get file ./data/sample_submission.csv
(3982, 207)
get file ./data/test_features.csv
(3982, 876)
get file ./data/train_drug.csv
(23814, 2)
get file ./data/train_features.csv
(23814, 876)
get file ./data/train_targets_scored.csv
(23814, 207)
get file ./data/train_targets_nonscored.csv
(23814, 403)


In [5]:
print("Train features:", train_features.shape)

print("Test features:", test_features.shape)

print("Scored targets:", train_targets_scored.shape)

print("Non-scored targets:", train_targets_nonscored.shape)

print("Drug information:", train_drug.shape)

print("\nFirst 5 rows:")
display(train_features.head())

print("\nTarget columns:")
print(train_targets_scored.columns.tolist()[:10])

print("\nMissing values:")
print(train_features.isnull().sum().sum())


Train features: (23814, 876)
Test features: (3982, 876)
Scored targets: (23814, 207)
Non-scored targets: (23814, 403)
Drug information: (23814, 2)

First 5 rows:


,sig_id,cp_type,cp_time,cp_dose,g-0,g-1,g-2,g-3,g-4,g-5,...,c-90,c-91,c-92,c-93,c-94,c-95,c-96,c-97,c-98,c-99
0,id_000644bb2,trt_cp,24,D1,1.0620,0.5577,-0.2479,-0.6208,-0.1944,-1.0120,...,0.2862,0.2584,0.8076,0.5523,-0.1912,0.6584,-0.3981,0.2139,0.3801,0.4176
1,id_000779bfc,trt_cp,72,D1,0.0743,0.4087,0.2991,0.0604,1.0190,0.5207,...,-0.4265,0.7543,0.4708,0.0230,0.2957,0.4899,0.1522,0.1241,0.6077,0.7371
2,id_000a6266a,trt_cp,48,D1,0.6280,0.5817,1.5540,-0.0764,-0.0323,1.2390,...,-0.7250,-0.6297,0.6103,0.0223,-1.3240,-0.3174,-0.6417,-0.2187,-1.4080,0.6931
3,id_0015fd391,trt_cp,48,D1,-0.5138,-0.2491,-0.2656,0.5288,4.0620,-0.8095,...,-2.0990,-0.6441,-5.6300,-1.3780,-0.8632,-1.2880,-1.6210,-0.8784,-0.3876,-0.8154
4,id_001626bd3,trt_cp,72,D2,-0.3254,-0.4009,0.9700,0.6919,1.4180,-0.8244,...,0.0042,0.0048,0.6670,1.0690,0.5523,-0.3031,0.1094,0.2885,-0.3786,0.7125



Target columns:
['sig_id', '5-alpha_reductase_inhibitor', '11-beta-hsd1_inhibitor', 'acat_inhibitor', 'acetylcholine_receptor_agonist', 'acetylcholine_receptor_antagonist', 'acetylcholinesterase_inhibitor', 'adenosine_receptor_agonist', 'adenosine_receptor_antagonist', 'adenylyl_cyclase_activator']

Missing values:
0


In [ ]:
target_columns = [
    col for col in train_targets_scored.columns
    if col != "sig_id"
]

y = train_targets_scored[target_columns].values

y = y.astype(np.float32)

print("Target shape:", y.shape)

print("Number of targets:", len(target_columns))

print("Unique labels:", np.unique(y))

display(y)


Target shape: (23814, 206)
Number of targets: 206
Unique labels: [0. 1.]


array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(23814, 206), dtype=float32)

In [ ]:
train_data = train_features[["sig_id"]].merge(
    train_drug,
    on="sig_id",
    how="left",
    validate="one_to_one"
)

groups = train_data["drug_id"].values

print("Number of samples:", len(groups))

print("Unique drugs:", pd.Series(groups).nunique())

Number of samples: 23814
Unique drugs: 3289


In [ ]:
train_ids = train_features["sig_id"].copy()
test_ids = test_features["sig_id"].copy()

X_train = train_features.drop(
    columns=["sig_id"]
).copy()

X_test = test_features.drop(
    columns=["sig_id"]
).copy()

all_features = pd.concat(
    [X_train, X_test],
    axis=0,
    ignore_index=True
)

categorical_columns = [
    "cp_type",
    "cp_time",
    "cp_dose"
]

for col in categorical_columns:
    all_features[col] = all_features[col].astype(str)

all_features = pd.get_dummies(
    all_features,
    columns=categorical_columns,
    dtype=np.float32
)

X_train = all_features.iloc[
    :len(train_features)
].copy()

X_test = all_features.iloc[
    len(train_features):
].copy()

print("Encoded train shape:", X_train.shape)
print("Encoded test shape:", X_test.shape)

numeric_cols = [c for c in X_train.columns if c.startswith('g-') or c.startswith('c-')]

scaler = StandardScaler()

X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

Encoded train shape: (23814, 879)
Encoded test shape: (3982, 879)


In [ ]:
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)

# 模型
class Ciallo_MLP(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(Ciallo_MLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.4),
            
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            
            nn.Linear(512, output_dim)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
BATCH_SIZE = 512
EPOCHS = 40
LR = 1e-3
DEVICE = device

gkf = GroupKFold(n_splits=5)

test_preds = np.zeros((len(X_test), len(target_columns)))
oof_preds = np.zeros((len(X_train), len(target_columns)))

input_dim = X_train.shape[1]
output_dim = len(target_columns)

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train_tensor, y_train_tensor, groups=groups)):
    print(f"\n========== Fold {fold+1} / 5 ==========")
    
    X_tr, y_tr = X_train_tensor[train_idx], y_train_tensor[train_idx]
    X_val, y_val = X_train_tensor[val_idx], y_train_tensor[val_idx]
    
    train_dataset = TensorDataset(X_tr, y_tr)
    val_dataset = TensorDataset(X_val, y_val)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE * 2, shuffle=False)
    
    model = Ciallo_MLP(input_dim, output_dim).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    
    best_val_loss = float('inf')
    best_model_wts = copy.deepcopy(model.state_dict())
    
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * batch_X.size(0)
            
        train_loss /= len(train_loader.dataset)
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item() * batch_X.size(0)
                
        val_loss /= len(val_loader.dataset)
        scheduler.step(val_loss)
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.5f} | Val Loss: {val_loss:.5f}")
            
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            
    print(f"Fold {fold+1} Best Val Loss: {best_val_loss:.5f}")
    
    model.load_state_dict(best_model_wts)
    model.eval()
    
    with torch.no_grad():
        val_outputs = model(X_val.to(DEVICE))
        oof_preds[val_idx] = torch.sigmoid(val_outputs).cpu().numpy()
        
        test_outputs = model(X_test_tensor.to(DEVICE))
        test_preds += torch.sigmoid(test_outputs).cpu().numpy() / 5.0

print("\n训练完成喵☆")


========== Fold 1 / 5 ==========
Epoch 01 | Train Loss: 0.15387 | Val Loss: 0.03069
Epoch 05 | Train Loss: 0.01832 | Val Loss: 0.01684
Epoch 10 | Train Loss: 0.01559 | Val Loss: 0.01559
Epoch 15 | Train Loss: 0.01351 | Val Loss: 0.01453
Epoch 20 | Train Loss: 0.01148 | Val Loss: 0.01455
Epoch 25 | Train Loss: 0.00958 | Val Loss: 0.01463
Epoch 30 | Train Loss: 0.00753 | Val Loss: 0.01443
Epoch 35 | Train Loss: 0.00551 | Val Loss: 0.01444
Epoch 40 | Train Loss: 0.00478 | Val Loss: 0.01445
Fold 1 Best Val Loss: 0.01427

========== Fold 2 / 5 ==========
Epoch 01 | Train Loss: 0.15343 | Val Loss: 0.04335
Epoch 05 | Train Loss: 0.01681 | Val Loss: 0.02824
Epoch 10 | Train Loss: 0.01399 | Val Loss: 0.02672
Epoch 15 | Train Loss: 0.01187 | Val Loss: 0.02528
Epoch 20 | Train Loss: 0.00978 | Val Loss: 0.02504
Epoch 25 | Train Loss: 0.00769 | Val Loss: 0.02466
Epoch 30 | Train Loss: 0.00551 | Val Loss: 0.02410
Epoch 35 | Train Loss: 0.00439 | Val Loss: 0.02467
Epoch 40 | Train Loss: 0.00394 | Va

In [ ]:
from sklearn.metrics import log_loss

train_ctl_mask = train_features["cp_type"] == "ctl_vehicle"
oof_preds[train_ctl_mask.values] = 1e-15

oof_preds_clipped = np.clip(oof_preds, 1e-15, 1 - 1e-15)
cv_score = log_loss(y.ravel(), oof_preds_clipped.ravel())
print(f"Overall CV Log Loss: {cv_score:.5f}")

test_ctl_mask = test_features["cp_type"] == "ctl_vehicle"
print(f"Number of ctl_vehicle in test: {test_ctl_mask.sum()}")

test_preds[test_ctl_mask.values, :] = 1e-15

submission = pd.DataFrame(test_preds, columns=target_columns)
submission.insert(0, "sig_id", test_ids)

submission.to_csv("submission.csv", index=False)
print("\nsubmission.csv 已生成喵☆")
print(submission.head())

Overall CV Log Loss: 0.01935
Number of ctl_vehicle in test: 358

submission.csv 已生成！
         sig_id  5-alpha_reductase_inhibitor  11-beta-hsd1_inhibitor  \
0  id_0004d9e33                 2.828570e-03            3.050974e-03   
1  id_001897cda                 1.455989e-03            2.670387e-03   
2  id_002429b5b                 1.000000e-15            1.000000e-15   
3  id_00276f245                 1.268785e-03            1.546275e-03   
4  id_0027f1083                 3.155305e-03            1.951316e-03   

   acat_inhibitor  acetylcholine_receptor_agonist  \
0    3.265965e-03                    2.350220e-02   
1    3.033516e-03                    3.376776e-03   
2    1.000000e-15                    1.000000e-15   
3    3.212804e-03                    7.165107e-03   
4    2.889562e-03                    1.894885e-02   

   acetylcholine_receptor_antagonist  acetylcholinesterase_inhibitor  \
0                       1.749891e-02                    9.757600e-03   
1                  